# Sovereign AI Engineering Lab

**Цель:** Разработать ядро Headless AI системы. Мы не используем готовые фреймворки (LangChain/LlamaIndex), чтобы понять математику и архитектуру "под капотом": от извлечения контекста через абстрактные синтаксические деревья до вызова инструментов по протоколу MCP и изоляции кода.

---

### ⚙️ Шаг 0: Настройка окружения и запуск локального инференса (Ollama)
Устанавливаем зависимости, запускаем сервер инференса в фоне и скачиваем легковесную специализированную модель для кодинга (Qwen 2.5 Coder), которая поместится в бесплатный GPU Colab.

In [ ]:
# 1. Установка библиотек
!pip install -q sentence-transformers ollama
!apt-get update && apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

import os
import time
import subprocess
import ast
import json
import inspect
import ollama
from sentence_transformers import SentenceTransformer, util

# 2. Запуск Ollama в фоновом режиме
print("Запускаем сервер Ollama...")
process = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5) # Ждем инициализации сервера

# 3. Скачиваем модель (выбрана версия 1.5b для скорости в Colab, в проде используем 7b-q8_0)
print("Скачиваем модель qwen2.5-coder:1.5b (это займет около минуты)...")
!ollama pull qwen2.5-coder:1.5b
print("Окружение готово!")

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [85.2 kB]
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,927 kB]
Get:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [6,652 kB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-updates/multiverse amd64 Packages [70.9 k

### Шаг 1: Context Engine (AST-Парсер)
*Отсылка к Слайду 15.* Чтобы не отправлять в модель мусор (и экономить токены), мы создаем движок, который читает код как структуру, извлекая только "скелеты" функций.

In [ ]:
# Создаем тестовый файл с кодом (имитация репозитория)
with open("payment_service.py", "w") as f:
    f.write("""
def process_payment(user_id, amount):
    \"\"\"Обрабатывает платеж пользователя.\"\"\"
    # БАГ: нет проверки на отрицательную сумму
    db.charge(user_id, amount)
    return True

def get_balance(user_id):
    \"\"\"Возвращает текущий баланс.\"\"\"
    return db.get_user(user_id).balance
""")

class ContextIndexer(ast.NodeVisitor):
    def __init__(self):
        self.functions =[]

    def visit_FunctionDef(self, node):
        # Извлекаем сигнатуры и документацию
        docstring = ast.get_docstring(node) or "Нет описания"
        args =[arg.arg for arg in node.args.args]
        self.functions.append({
            "name": node.name,
            "signature": f"{node.name}({', '.join(args)})",
            "doc": docstring,
            "lineno": node.lineno
        })
        self.generic_visit(node)

# Индексируем файл
indexer = ContextIndexer()
with open("payment_service.py", "r") as f:
    indexer.visit(ast.parse(f.read()))

print("Извлеченный контекст (Скелет кода):")
print(json.dumps(indexer.functions, indent=2, ensure_ascii=False))

Извлеченный контекст (Скелет кода):
[
  {
    "name": "process_payment",
    "signature": "process_payment(user_id, amount)",
    "doc": "Обрабатывает платеж пользователя.",
    "lineno": 2
  },
  {
    "name": "get_balance",
    "signature": "get_balance(user_id)",
    "doc": "Возвращает текущий баланс.",
    "lineno": 8
  }
]


### Шаг 2: Model Context Protocol (MCP) и Tool Calling
*Отсылка к Слайду 21.* Агент не знает нашу инфраструктуру. MCP-сервер динамически собирает доступные инструменты и отдает их агенту в виде JSON Schema.

In [ ]:
# 1. Определяем инструменты корпоративной инфраструктуры
def execute_sql(query: str) -> str:
    """Выполняет SQL запрос к базе данных (Read-Only)."""
    return f"Mock Result for {query}:[{'id': 1, 'balance': 100}]"

def get_jira_ticket(ticket_id: str) -> str:
    """Получает описание задачи из Jira."""
    return f"Ticket {ticket_id}: Fix negative amount bug in payments."

tools_registry = {
    "execute_sql": execute_sql,
    "get_jira_ticket": get_jira_ticket
}

# 2. Имитация MCP-сервера: автоматическая генерация схем
def mcp_discover_tools():
    mcp_schema =[]
    for name, func in tools_registry.items():
        sig = inspect.signature(func)
        params = {
            p_name: {"type": "string" if p.annotation == str else "any"}
            for p_name, p in sig.parameters.items()
        }
        mcp_schema.append({
            "type": "function",
            "function": {
                "name": name,
                "description": func.__doc__,
                "parameters": {
                    "type": "object",
                    "properties": params,
                    "required": list(params.keys())
                }
            }
        })
    return mcp_schema

print("MCP Schema, передаваемая агенту:")
print(json.dumps(mcp_discover_tools(), indent=2, ensure_ascii=False))

MCP Schema, передаваемая агенту:
[
  {
    "type": "function",
    "function": {
      "name": "execute_sql",
      "description": "Выполняет SQL запрос к базе данных (Read-Only).",
      "parameters": {
        "type": "object",
        "properties": {
          "query": {
            "type": "string"
          }
        },
        "required": [
          "query"
        ]
      }
    }
  },
  {
    "type": "function",
    "function": {
      "name": "get_jira_ticket",
      "description": "Получает описание задачи из Jira.",
      "parameters": {
        "type": "object",
        "properties": {
          "ticket_id": {
            "type": "string"
          }
        },
        "required": [
          "ticket_id"
        ]
      }
    }
  }
]


### Шаг 3: Инженерия намерений (`.clinerules`)
*Отсылка к Слайдам 17-18.* Агент работает хаотично без жестких рамок. Мы пишем правила, которые внедряются в системный промпт.

In [ ]:
# Создаем файл правил проекта
with open(".clinerules", "w") as f:
    f.write("""
- [CRITICAL] В финансовых функциях всегда проверяй входные данные на валидность (amount > 0).
- [STYLE] Используй type hints (typing) для всех аргументов функций.
- [SECURITY] Запрещено логировать имена пользователей.
""")

def build_system_prompt(role_description):
    with open(".clinerules", "r") as f:
        rules = f.read()
    return f"{role_description}\n\nКОРПОРАТИВНЫЕ ПРАВИЛА (ОБЯЗАТЕЛЬНО К ИСПОЛНЕНИЮ):\n{rules}"

system_prompt = build_system_prompt("Ты Senior Python Backend Developer.")
print(system_prompt)

Ты Senior Python Backend Developer.

КОРПОРАТИВНЫЕ ПРАВИЛА (ОБЯЗАТЕЛЬНО К ИСПОЛНЕНИЮ):

- [CRITICAL] В финансовых функциях всегда проверяй входные данные на валидность (amount > 0).
- [STYLE] Используй type hints (typing) для всех аргументов функций.
- [SECURITY] Запрещено логировать имена пользователей.



### 🛡️ Шаг 4: Безопасная песочница (Sandboxing & Guardrails)
*Отсылка к Слайду 24.* Если ИИ генерирует код, мы обязаны проверить его до исполнения на наличие деструктивных вызовов (RCE).

In [ ]:
class SecurityFilter(ast.NodeVisitor):
    def __init__(self):
        self.is_safe = True
        self.errors =[]

    def visit_Import(self, node):
        for alias in node.names:
            if alias.name in ['os', 'subprocess', 'sys', 'socket']:
                self.is_safe = False
                self.errors.append(f"Запрещенный импорт: {alias.name}")
        self.generic_visit(node)

def safe_sandbox_executor(code_str):
    try:
        tree = ast.parse(code_str)
        sec_filter = SecurityFilter()
        sec_filter.visit(tree)

        if not sec_filter.is_safe:
            return f"GUARDRAIL BLOCK: {', '.join(sec_filter.errors)}"

        # Безопасное исполнение в изолированном пространстве имен
        local_env = {}
        exec(code_str, {"__builtins__": {}}, local_env)
        return f"Успешно исполнено. Локальные переменные: {list(local_env.keys())}"
    except Exception as e:
        return f"Синтаксическая ошибка: {e}"

# Проверим:
print("Тест безопасного кода:", safe_sandbox_executor("x = 10 + 5"))
print("Тест опасного кода:", safe_sandbox_executor("import os\nos.system('rm -rf /')"))

Тест безопасного кода: Успешно исполнено. Локальные переменные: ['x']
Тест опасного кода: GUARDRAIL BLOCK: Запрещенный импорт: os


### Шаг 5: Grand Finale — Полный цикл Sovereign AI Агента
Собираем все компоненты вместе. Агент получает задачу, читает `.clinerules`, анализирует AST-скелет и использует локальную модель Qwen для генерации безопасного исправления.

In [ ]:
def run_autonomous_agent(task):
    # 1. Применяем правила (Шаг 3)
    sys_prompt = build_system_prompt("Ты AI-инженер. Твоя задача исправить баг в коде.")

    # 2. Собираем контекст через AST (Шаг 1)
    context_data = json.dumps(indexer.functions, ensure_ascii=False)

    user_prompt = f"""
    Задача: {task}
    Доступный код проекта (сигнатуры): {context_data}

    Напиши ИСПРАВЛЕННУЮ функцию process_payment целиком.
    Верни ТОЛЬКО Python код, без markdown разметки и без лишних слов.
    """

    # 3. Вызов локальной модели (Инференс)
    print("Отправка запроса в Ollama...")
    response = ollama.chat(model='qwen2.5-coder:1.5b', messages=[
        {'role': 'system', 'content': sys_prompt},
        {'role': 'user', 'content': user_prompt}
    ])

    generated_code = response['message']['content'].strip().replace("```python", "").replace("```", "")
    print("\n--- Сгенерированный код ---")
    print(generated_code)
    print("---------------------------\n")

    # 4. Проверка в песочнице (Шаг 4)
    print("Аудит безопасности (Sandbox):")
    audit_result = safe_sandbox_executor(generated_code)
    print(audit_result)

# Запускаем пайплайн!
run_autonomous_agent("В функции process_payment можно передать отрицательную сумму. Исправь баг с учетом корпоративных правил (type hints, проверка > 0).")

Отправка запроса в Ollama...

--- Сгенерированный код ---

def process_payment(user_id: int, amount: float) -> None:
    if amount <= 0:
        raise ValueError("Amount must be greater than 0.")
    # Logic for processing the payment
    print(f"Processing payment for user {user_id} with amount {amount}")


В этом исправлении:
1. Указана типизация аргументов функции (`user_id` и `amount`).
2. Валидация входного параметра `amount` на положительность.
3. Если `amount` не положительное, выбрасывается исключение `ValueError`.
4. Выводится сообщение об обработке платежа.
---------------------------

Аудит безопасности (Sandbox):
Синтаксическая ошибка: invalid syntax (<unknown>, line 9)


### Шаг 6: A2A (Agent-to-Agent) и LLM-as-a-Judge (Оценка)
*Отсылка к Слайдам 22-23.* В проде результат работы Кодера всегда оценивает Ревьюер. Мы используем векторные эмбеддинги для оценки семантического сходства (Context Relevance).

In [ ]:
embedder = SentenceTransformer('all-MiniLM-L6-v2')

def llm_as_a_judge_evaluate(reference_text, generated_text):
    """Оценивает, насколько сгенерированный код близок к ожидаемому смыслу."""
    embeddings = embedder.encode([reference_text, generated_text])
    # Косинусное сходство между эталоном и результатом
    score = util.cos_sim(embeddings[0], embeddings[1]).item()
    return score

эталон = """
def process_payment(user_id: int, amount: float) -> bool:
    if amount <= 0:
        raise ValueError("Amount must be positive")
    db.charge(user_id, amount)
    return True
"""

тест_код = """
def process_payment(user_id, amount):
    if amount > 0:
        db.charge(user_id, amount)
        return True
    return False
"""

similarity = llm_as_a_judge_evaluate(эталон, тест_код)
print(f"Оценка качества кода (Семантическое сходство): {similarity:.2f} / 1.00")
if similarity < 0.8:
    print("Вердикт Ревьюера: Код требует доработки (не совпадает с эталонной логикой).")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Оценка качества кода (Семантическое сходство): 0.93 / 1.00


### Практические задания для студентов (Самостоятельная работа)

**Задание 1: Интеграция MCP в пайплайн (Tool Calling)**
Сейчас в Шаге 5 мы просто отдаем код.
*Задача:* Измените системный промпт в Шаге 5, передав агенту JSON-схему инструментов из Шага 2 (`mcp_discover_tools()`). Научите агента отвечать JSON-ом вида `{"tool": "get_jira_ticket", "ticket_id": "123"}`. Напишите обработчик (If/Else), который перехватит этот JSON, вызовет функцию из `tools_registry` и вернет агенту результат.

**Задание 2: Продвинутый AST-Context (Граф вызовов)**
В Шаге 1 мы извлекаем только имена функций.
*Задача:* Добавьте в класс `ContextIndexer` метод `visit_Call(self, node)`. Внутри него находите все узлы `ast.Name`, чтобы понять, **какие другие функции** вызывает текущая функция. Сохраняйте этот список в словарь. Это научит вас строить граф зависимостей (Call Graph).

**Задание 3: Эволюция Guardrails (DLP Фильтр)**
В Шаге 4 мы ловим только импорты.
*Задача:* Допишите в `safe_sandbox_executor` проверку исходного текста кода (строки `code_str`) с помощью модуля `re` (RegEx). Если в сгенерированном коде встречается паттерн "токен" (например, `api_key = "sk-..."`), песочница должна блокировать код с ошибкой "DLP Alert: Обнаружен хардкод секретов".

<details>
<summary><b>Показать решение </b></summary>

```python
import ast
import json
import re
import inspect


# ЗАДАНИЕ 2: Продвинутый AST-Context (Граф вызовов / Call Graph)

print("=== ЗАДАНИЕ 2: ГРАФ ВЫЗОВОВ (AST) ===")

# Тестовый код, в котором одна функция вызывает другие
source_code = """
def process_payment(user_id, amount):
    if amount <= 0:
        raise ValueError("Invalid amount")
    db.charge(user_id, amount)
    send_receipt(user_id)
    return True
"""

class AdvancedContextIndexer(ast.NodeVisitor):
    def __init__(self):
        self.functions =[]

    def visit_FunctionDef(self, node):
        calls = set() # Используем set, чтобы избежать дубликатов

        # Обходим все узлы ВНУТРИ текущей функции
        for child in ast.walk(node):
            if isinstance(child, ast.Call):
                # Если это прямой вызов, например send_receipt()
                if isinstance(child.func, ast.Name):
                    calls.add(child.func.id)
                # Если это вызов метода объекта, например db.charge()
                elif isinstance(child.func, ast.Attribute):
                    calls.add(child.func.attr)

        self.functions.append({
            "name": node.name,
            "calls_to": list(calls) # Сохраняем граф зависимостей
        })
        self.generic_visit(node)

advanced_indexer = AdvancedContextIndexer()
advanced_indexer.visit(ast.parse(source_code))

print("Извлеченный граф зависимостей:")
print(json.dumps(advanced_indexer.functions, indent=2, ensure_ascii=False))
print("-" * 50)


# ЗАДАНИЕ 3: Эволюция Guardrails (DLP Фильтр / RegEx)

print("\n=== ЗАДАНИЕ 3: DLP ФИЛЬТР (УТЕЧКА СЕКРЕТОВ) ===")

def dlp_sandbox_executor(code_str):
    # Паттерн ищет хардкод токенов: api_key = "sk-...", password="...", token='...'
    # (?i) делает поиск нечувствительным к регистру
    secret_pattern = re.compile(
        r"(?i)(api_key|password|token|secret)\s*=\s*['\"](sk-[a-zA-Z0-9]{20,}|[a-zA-Z0-9]{10,})['\"]"
    )

    # 1. Сначала проверяем текст регулярным выражением (DLP)
    match = secret_pattern.search(code_str)
    if match:
        return f"DLP ALERT: Заблокировано! Обнаружен хардкод секрета в переменной '{match.group(1)}'."

    # 2. Если секретов нет, проверяем AST (как в Шаге 4 лекции)
    try:
        tree = ast.parse(code_str)
        return "Sandbox: Код безопасен. Утечек не обнаружено, синтаксис верен."
    except Exception as e:
        return f"Синтаксическая ошибка: {e}"

# Тестируем безопасный и опасный код
safe_code = "api_key = os.getenv('API_KEY')" # Берем из окружения - это безопасно
danger_code = "api_key = 'sk-1234567890abcdefGHiklmnopqrstuvwxyz'" # Хардкод - опасно!

print(f"Тест 1 (Безопасный код): {dlp_sandbox_executor(safe_code)}")
print(f"Тест 2 (Опасный код): {dlp_sandbox_executor(danger_code)}")
print("-" * 50)


# ЗАДАНИЕ 3: Интеграция MCP в пайплайн (Tool Calling Dispatcher)

print("\n=== ЗАДАНИЕ 1: ИНТЕГРАЦИЯ MCP (TOOL DISPATCHER) ===")

# 1. Наши инструменты (Tools)
def get_jira_ticket(ticket_id: str) -> str:
    return f"[JIRA] Задача {ticket_id}: Пользователи жалуются на отрицательный баланс."

tools_registry = {
    "get_jira_ticket": get_jira_ticket
}

# 2. Диспетчер (Оркестратор), который парсит ответ LLM и вызывает нужный Python-код
def mcp_tool_dispatcher(llm_response_text: str):
    print(f"Ответ агента (LLM): {llm_response_text}")

    # Пытаемся вытащить JSON из ответа (защита от "грязного" Markdown)
    try:
        # Ищем начало и конец JSON
        start_idx = llm_response_text.find('{')
        end_idx = llm_response_text.rfind('}') + 1

        if start_idx == -1 or end_idx == 0:
            return "Ошибка: Агент не вернул JSON."

        json_str = llm_response_text[start_idx:end_idx]
        call_data = json.loads(json_str)

        tool_name = call_data.get("tool")
        tool_args = call_data.get("args", {})

        # Проверяем, существует ли такой инструмент
        if tool_name not in tools_registry:
            return f"Ошибка: Инструмент '{tool_name}' не найден в MCP-реестре."

        # Вызываем реальную функцию Python с аргументами из JSON
        print(f"Вызов инструмента: {tool_name}(**{tool_args})")
        result = tools_registry[tool_name](**tool_args)

        return f"Результат выполнения инструмента: {result}"

    except json.JSONDecodeError:
        return "Ошибка: Сгенерирован невалидный JSON. Требуется повторный вызов (Retry)."
    except Exception as e:
        return f"Ошибка исполнения инструмента: {e}"

# Имитируем идеальный ответ от языковой модели
llm_perfect_response = """
Мне нужно прочитать задачу, чтобы понять контекст.
{
  "tool": "get_jira_ticket",
  "args": {"ticket_id": "BACKEND-404"}
}
"""

# Имитируем ответ модели с галлюцинацией (выдумала инструмент)
llm_hallucination_response = """
{"tool": "delete_database", "args": {"force": true}}
"""

print("\n--- Сценарий А (Успешный вызов) ---")
print(mcp_tool_dispatcher(llm_perfect_response))

print("\n--- Сценарий Б (Защита от галлюцинаций) ---")
print(mcp_tool_dispatcher(llm_hallucination_response))
```
</details>